# Extracting text from documents in various formats
---

## Installing the dependencies
---

In [ ]:
# @title
!pip install markitdown[pdf] --quiet
!pip install PyPDF2 --quiet
!pip install pymupdf4llm --quiet
!pip install easyocr --quiet
!pip install tiktoken --quiet
!pip install verovio --quiet
!pip install mistralai --quiet
!pip install -U openai-whisper --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Extracting text from PDF documents
---

In [ ]:
# Button to upload a file at the start of the notebook
from google.colab import files
uploaded = files.upload()

In [ ]:
# Path to the PDF file
pdf_path = "/content/Welcome_notice_Willow_Creek_City_Hall.pdf"

### Markitdown

In [ ]:
from markitdown import MarkItDown

md = MarkItDown()
result = md.convert(pdf_path)
print(result.text_content)

### PyPDF2

In [ ]:
import PyPDF2

# Open the PDF file in binary read mode
with open(pdf_path, "rb") as pdf_file:
    # Create a PDF reader
    pdf_reader = PyPDF2.PdfReader(pdf_file)

    # Extract the text from each page
    text = ""
    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        text += page.extract_text()

    # Display the extracted text
    print(text)

### Pymupdf4llm

In [ ]:
import pymupdf4llm

md_text = pymupdf4llm.to_markdown(pdf_path)
print("=== Markdown content extracted with PyMuPDF4LLM ===")
print(md_text)

## OCR the text contained in images

---


In [ ]:
# Button to upload a file at the start of the notebook
from google.colab import files
uploaded = files.upload()

In [ ]:
image_path = "/content/willow_creek_christmas_market_2023.png"

### Easy OCR

In [ ]:
import easyocr
import os

# Initialize EasyOCR
reader = easyocr.Reader(['en'])  # 'en' for English, add other languages if needed

# Extract the text from each image
full_text = ""

# Apply OCR on the image
results = reader.readtext(image_path)

# Format the results
for (bbox, text, prob) in results:
    full_text += text + "\n"
print(full_text)

### GOT OCR

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True)
model = AutoModel.from_pretrained('ucaslcl/GOT-OCR2_0', trust_remote_code=True, low_cpu_mem_usage=True, device_map='cuda', use_safetensors=True, pad_token_id=tokenizer.eos_token_id)

# plain texts OCR
res = model.chat(tokenizer, image_path, ocr_type='ocr')

print(res)

### Mistral Document Understanding

In [ ]:
import base64
import requests
import os
from mistralai import Mistral

# Read the API key from an environment variable (never hard-code a secret)
api_key = os.environ.get("MISTRAL_API_KEY", "YOUR_MISTRAL_API_KEY")



def encode_image(image_path):
    """Encode the image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: The file {image_path} was not found.")
        return None
    except Exception as e:  # Added general exception handling
        print(f"Error: {e}")
        return None

# Getting the base64 string
base64_image = encode_image(image_path)

client = Mistral(api_key=api_key)

ocr_response = client.ocr.process(
    model="mistral-ocr-latest",
    document={
        "type": "image_url",
        "image_url": f"data:image/jpeg;base64,{base64_image}"
    }
)
ocr_response

## Extracting text from audio documents

---


In [ ]:
# Button to upload a file at the start of the notebook
from google.colab import files
uploaded = files.upload()

In [ ]:
# Path to the audio file
audio_path = "/content/willow_creek_greetings_2025.wav"

### Whisper

In [ ]:
import whisper

# Transcribe with Whisper
def transcribe_audio_with_whisper(audio_file, model_name="base"):
    # Load the Whisper model
    model = whisper.load_model(model_name)

    # Transcribe the audio file
    result = model.transcribe(audio_file)

    # Display the transcription
    print("Transcription:")
    print(result["text"])

# Transcribe the audio with Whisper
transcribe_audio_with_whisper(audio_path)